# GIK-IceChain v2.0 — E2E Pipeline Test (Colab)

This notebook reproduces the full **C1 → C2 → C3** pipeline from
[`run_e2e_vps.sh`](https://github.com/hashirama21/gik-icechain/blob/main/run_e2e_vps.sh)
on Google Colab.

| Phase | What it does |
|-------|-------------|
| **1** | System setup (uv, eccodes, Python check) |
| **2** | Clone repo + install dependencies |
| **3** | Download reference data + pre-fitted GEV thresholds |
| **4** | Run `gik-icechain run-all` (C1 → C2 → C3) |
| **5** | Verify outputs + display risk summary |

> **Runtime**: Use a standard CPU runtime. No GPU needed.

---
## Phase 1 — System Setup

In [ ]:
import sys, shutil, subprocess

# Python version check
py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
assert sys.version_info >= (3, 12), f"Python >= 3.12 required, got {py_ver}"
print(f"OK  Python {py_ver}")

# Install uv (fast Python package manager)
if not shutil.which("uv"):
    print("INFO Installing uv ...")
    subprocess.check_call(["pip", "install", "uv"], stdout=subprocess.DEVNULL)
print(f"OK  uv installed")

# Install eccodes system library (GRIB2 decoding)
print("INFO Installing eccodes system library ...")
subprocess.check_call(
    ["apt-get", "update", "-qq"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
subprocess.check_call(
    ["apt-get", "install", "-y", "-qq", "libeccodes0", "libeccodes-tools"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("OK  eccodes")

---
## Phase 2 — Clone Repository + Install Dependencies

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/hashirama21/gik-icechain.git"
BRANCH   = "develop"
REPO_DIR = Path("/content/gik-icechain")

if not (REPO_DIR / "pyproject.toml").exists():
    print(f"INFO Cloning repository (branch: {BRANCH}) ...")
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
    print("OK  Repository cloned")
else:
    print("OK  Repository already present")

os.chdir(REPO_DIR)
print(f"OK  Working directory: {Path.cwd()}")

In [ ]:
print("INFO Running uv sync (this installs all dependencies) ...")
!uv sync 2>&1 | tail -5

# Verify CLI is accessible
!uv run gik-icechain --help > /dev/null 2>&1 && echo "OK  CLI 'gik-icechain' accessible" || echo "FAIL  CLI not working"

---
## Phase 3 — Reference Data Download

In [ ]:
# Admin boundaries + raw CMORPH + ENSO/IOD index
print("INFO Downloading admin boundaries, raw CMORPH, ENSO/IOD ...")
!uv run python3 scripts/tools.py download --component all

# Pre-fitted GEV thresholds (required by C2)
print("\nINFO Downloading pre-fitted GEV thresholds ...")
!uv run python3 scripts/tools.py download-thresholds

---
## Phase 4 — E2E Pipeline (C1 → C2 → C3)

In [ ]:
# Pipeline parameters
START_DATE = "2025-10-24"
END_DATE   = "2025-10-30"
CONFIG     = "configs/default.yaml"
OUTPUT     = "results/e2e_colab"

# MinIO dev credentials
os.environ["AWS_ENDPOINT_URL"]    = "http://20.116.218.195:9000"
os.environ["AWS_ACCESS_KEY_ID"]     = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["AWS_REGION"]            = "us-east-1"

print(f"Date range : {START_DATE} -> {END_DATE}")
print(f"Config     : {CONFIG}")
print(f"Output     : {OUTPUT}")
print(f"MinIO      : {os.environ['AWS_ENDPOINT_URL']}")

In [ ]:
import shutil

# Clean previous run
output_path = Path(OUTPUT)
if output_path.exists():
    shutil.rmtree(output_path)
output_path.mkdir(parents=True, exist_ok=True)

print("INFO Running: gik-icechain run-all")
print(f"     --start  {START_DATE}")
print(f"     --end    {END_DATE}")
print(f"     --config {CONFIG}")
print(f"     --output {OUTPUT}")
print()

!ECCODES_PYTHON_USE_FINDLIBS=1 uv run gik-icechain run-all \
    --start {START_DATE} \
    --end {END_DATE} \
    --config {CONFIG} \
    --output {OUTPUT}

---
## Phase 5 — Verification

In [ ]:
import glob, json

risk_dir = Path(OUTPUT) / "admin1_risk"
files = sorted(glob.glob(str(risk_dir / "*_risk_scores.json")))

if not files:
    print("FAIL  No risk score files produced.")
else:
    print(f"OK  Risk scores: {len(files)} file(s) in {risk_dir}")
    print()
    print(f"  Days processed: {len(files)}")
    print(f"  {'Date':<14} {'Units':>6}  Risk distribution")
    print(f"  {'---'*5:<14} {'---':>6}  {'---'*10}")

    for f in files:
        data = json.load(open(f))
        units = data.get("units", {})
        counts = {}
        for u in units.values():
            lbl = u.get("risk_label", "?")
            counts[lbl] = counts.get(lbl, 0) + 1
        dist = ", ".join(f"{k}: {v}" for k, v in sorted(counts.items()))
        print(f"  {data.get('date', '?'):<14} {len(units):>6}  {dist}")

In [ ]:
print("========================================")
print(" Summary")
print("========================================")
print(f"  Date range : {START_DATE} -> {END_DATE}")
print(f"  Config     : {CONFIG}")
print(f"  Output     : {OUTPUT}")
print(f"  MinIO      : {os.environ['AWS_ENDPOINT_URL']}")
print()
if files:
    print("All phases passed.")
else:
    print("Pipeline failed. Review output above.")